# CNNs — Version 2: Generalization, regularization, and noisy images

**IMT · Introduction to Machine Learning · Google Colab**

**Level:** a small step beyond the first CNN lab. Work individually or in pairs.

**Prerequisite:** complete Version 1 (convolution, pooling, training, accuracy). We reuse the **same 8 × 8 digit dataset**, so there is no download or extra package installation.

**Core exercises:** 1) adapt a CNN to use dropout; 2) distinguish training/validation/test; 3) explore overfitting with little data; 4) test robustness to noise.

**Bonus exercises:** inspect mistakes; train with noisy data augmentation; add simple early stopping.

**Estimated time:** ~40–55 minutes for core exercises, plus bonuses. Cells marked `TODO` are intentional stopping points; replace the `raise NotImplementedError(...)` lines with your code. Run cells top to bottom on a CPU runtime.

## Setup — three different datasets, three different purposes

- **Training set:** updates the model weights.
- **Validation set:** compares design choices such as dropout and number of epochs.
- **Test set:** kept unseen until the final check of a chosen model.

We'll first split digits into **training / validation / test** sets (approximately 60% / 20% / 20%). Later, we also make a tiny training subset to demonstrate overfitting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay

SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)

images = load_digits().images.astype("float32") / 16.0
labels = load_digits().target.astype("int64")

# First reserve 20% for the untouched test set.
X_tmp, X_test, y_tmp, y_test = train_test_split(
    images, labels, test_size=0.20, stratify=labels, random_state=SEED
)
# Then reserve 25% of the remaining 80% for validation: 60/20/20 overall.
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=SEED
)

def as_tensors(X, y):
    return (torch.tensor(X[:, None, :, :], dtype=torch.float32),
            torch.tensor(y, dtype=torch.long))

xtr, ytr = as_tensors(X_train, y_train)
xval, yval = as_tensors(X_val, y_val)
xte, yte = as_tensors(X_test, y_test)
train_loader = DataLoader(TensorDataset(xtr, ytr), batch_size=64, shuffle=True)

# 150 examples (15 per digit), used later to *deliberately* create a low-data regime.
small_ids = np.concatenate([np.flatnonzero(y_train == digit)[:15] for digit in range(10)])
xsmall, ysmall = xtr[small_ids], ytr[small_ids]
small_loader = DataLoader(TensorDataset(xsmall, ysmall), batch_size=50, shuffle=True)

print(f"Training: {len(xtr)} | validation: {len(xval)} | test: {len(xte)}")
print(f"Low-data training subset: {len(xsmall)} examples ({len(xsmall)//10} per digit)")
fig, axs = plt.subplots(2, 5, figsize=(8, 3.4))
for ax, img, digit in zip(axs.flat, X_train[:10], y_train[:10]):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set(title=f"Label {digit}", xticks=[], yticks=[])
plt.tight_layout(); plt.show()

## ✍️ Exercise 1 — A CNN with optional dropout

Define `DigitCNN` with the following structure:

1. `Conv2d(1, 8, 3, padding=1)` → ReLU → `MaxPool2d(2)`
2. `Conv2d(8, 16, 3, padding=1)` → ReLU → `MaxPool2d(2)`
3. `Flatten()` → `Dropout(p=dropout_rate)` → `Linear(16*2*2, 10)`

**Questions:** What is the shape after each pooling layer? What does dropout do during training? Does it change anything during evaluation (`model.eval()`)?

**Hint:** use `nn.Sequential(...)`, as in Version 1. Do **not** add softmax at the end.

In [ ]:
class DigitCNN(nn.Module):
    def __init__(self, dropout_rate=0.0):
        super().__init__()
        # TODO: define self.net, using dropout_rate in nn.Dropout(...).
        raise NotImplementedError("Build a 2-layer CNN with configurable dropout")

    def forward(self, x):
        return self.net(x)

check_model = DigitCNN(dropout_rate=0.25)
assert check_model(torch.zeros(4, 1, 8, 8)).shape == (4, 10)
print(check_model)
print("Trainable parameters:", sum(p.numel() for p in check_model.parameters()))

## Training helper (provided)

To focus on new ML concepts, the loop below is already written. It uses:

- `CrossEntropyLoss` and Adam,
- a **training** pass to update weights,
- a **validation** pass with `model.eval()` and `torch.no_grad()`,
- a stored history of training and validation accuracy.

Read through the helper and find where the weights are updated and where they are **not** updated.

In [ ]:
@torch.no_grad()
def accuracy_on(model, x, y):
    model.eval()
    logits = model(x)
    return (logits.argmax(dim=1) == y).float().mean().item()


def fit(model, loader, x_valid, y_valid, epochs=18, learning_rate=0.005,
        train_noise_std=0.0, verbose=False, patience=None):
    """Train on loader, evaluate on validation; optionally augment input with noise.

    If patience is set, stop after that many epochs without validation-loss improvement
    and restore the best weights (a BONUS feature; default is off).
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss()
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_loss = float("inf")
    best_weights = None
    bad_epochs = 0

    for epoch in range(epochs):
        model.train()
        total_loss, total_correct, seen = 0.0, 0, 0
        for xb, yb in loader:
            if train_noise_std > 0:
                xb = (xb + train_noise_std * torch.randn_like(xb)).clamp(0.0, 1.0)
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(yb)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            seen += len(yb)

        model.eval()
        with torch.no_grad():
            v_logits = model(x_valid)
            v_loss = loss_fn(v_logits, y_valid).item()
            v_acc = (v_logits.argmax(dim=1) == y_valid).float().mean().item()
        history["train_loss"].append(total_loss / seen)
        history["train_acc"].append(total_correct / seen)
        history["val_loss"].append(v_loss)
        history["val_acc"].append(v_acc)

        if v_loss < best_val_loss - 1e-5:
            best_val_loss = v_loss
            best_weights = {name: weight.clone() for name, weight in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
        if verbose and (epoch == 0 or (epoch + 1) % 5 == 0):
            print(f"Epoch {epoch+1:02d}: train {history['train_acc'][-1]:.1%} | val {v_acc:.1%}")
        if patience is not None and bad_epochs >= patience:
            print(f"Early stopping at epoch {epoch+1}; best val loss = {best_val_loss:.4f}")
            model.load_state_dict(best_weights)
            break

    return model, history


def plot_histories(histories, metric="acc"):
    """histories is a dict: label -> history."""
    plt.figure(figsize=(7, 3.6))
    for label, history in histories.items():
        plt.plot(history[f"train_{metric}"], label=f"{label} train")
        plt.plot(history[f"val_{metric}"], "--", label=f"{label} validation")
    plt.xlabel("Epoch (starting at 0)")
    plt.ylabel("Accuracy" if metric == "acc" else "Loss")
    plt.legend(); plt.grid(alpha=0.2); plt.tight_layout(); plt.show()

## ✍️ Exercise 2 — Train on the full dataset

Start with **no dropout**. Train on the **training set**, and monitor the **validation** accuracy. You should not examine the test accuracy to choose model hyperparameters.

**Complete the `TODO`**: create `full_model = DigitCNN(dropout_rate=0.0)`, then call `fit(full_model, train_loader, xval, yval, epochs=15)`.

**Questions:** Why does validation accuracy matter? Is it always smaller than training accuracy?

In [ ]:
torch.manual_seed(SEED)
# TODO: instantiate full_model (0 dropout), then train with fit(...).
raise NotImplementedError("Train the CNN on the full training set")

plot_histories({"Full-data CNN": full_history})
print("Final validation accuracy:", f"{full_history['val_acc'][-1]:.1%}")

## ✍️ Exercise 3 — What changes when we have very little data?

We created a balanced subset of only **150 training digits**, but we still validate on **all** validation examples.

1. Train a **fresh** model without dropout on `small_loader` for **35 epochs**.
2. Plot and compare training and validation accuracy.
3. Train another fresh model with `dropout_rate=0.35` on the same 150 digits. Keep other settings the same.

**Question:** What is *overfitting*? Did dropout help validation accuracy in this run? (It may or may not; that is a legitimate result.)

**Fairness tip:** reset the random seed before creating each model.

In [ ]:
torch.manual_seed(SEED)
# TODO: train small_model with dropout_rate=0.0 for 35 epochs on small_loader.
raise NotImplementedError("Train the low-data CNN without dropout")

# TODO: reset the seed and train dropout_model with dropout_rate=0.35.
raise NotImplementedError("Train the low-data CNN with dropout")

plot_histories({"Small, no dropout": small_history,
                "Small, dropout=0.35": dropout_history})
print(f"No dropout: train {small_history['train_acc'][-1]:.1%}, "
      f"validation {small_history['val_acc'][-1]:.1%}")
print(f"With dropout: train {dropout_history['train_acc'][-1]:.1%}, "
      f"validation {dropout_history['val_acc'][-1]:.1%}")

## ✍️ Exercise 4 — Will the CNN handle noisy images?

For a final check, use the **full-data model from Exercise 2** and the **held-out test set**. First evaluate clean test images. Next make a *corrupted* copy:

```python
xte_noisy = (xte + 0.35 * torch.randn_like(xte)).clamp(0, 1)
```

**Complete the TODO:** calculate `acc_clean` and `acc_noisy` using the provided `accuracy_on` function.

**Questions:** Did the model learn the digit or memorize exact pixel values? Why might noise hurt performance?

In [ ]:
# Fixed noise seed makes comparisons reproducible.
torch.manual_seed(123)
xte_noisy = (xte + 0.35 * torch.randn_like(xte)).clamp(0.0, 1.0)

# TODO: evaluate full_model on (xte, yte) and (xte_noisy, yte).
raise NotImplementedError("Calculate acc_clean and acc_noisy")

print(f"Clean test accuracy: {acc_clean:.1%}")
print(f"Noisy test accuracy: {acc_noisy:.1%}")

fig, axes = plt.subplots(2, 5, figsize=(8, 3.8))
for j in range(5):
    axes[0, j].imshow(xte[j, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, j].imshow(xte_noisy[j, 0], cmap="gray", vmin=0, vmax=1)
    for i in range(2): axes[i, j].axis("off")
axes[0, 0].set_ylabel("Clean")
axes[1, 0].set_ylabel("Noisy")
plt.suptitle("Same digits, different conditions")
plt.tight_layout(); plt.show()

## Bonus 1 — Diagnose mistakes visually

Find the **first 6 incorrectly classified noisy test digits** using `full_model`. Plot them with their true labels and predicted labels.

**Hint:** `pred = full_model(xte_noisy).argmax(dim=1)` inside `torch.no_grad()`; then use `np.flatnonzero(...)`.

**Question:** Are some digits especially sensitive to noise?

In [ ]:
# BONUS TODO: get predictions, find incorrect indices, and show up to 6 images.
raise NotImplementedError("Visualize the model's noisy-image mistakes")

## Bonus 2 — Data augmentation for noise robustness

The `fit` helper already supports `train_noise_std`.

1. Train a *fresh* model on the **full training set** with `train_noise_std=0.25` for 15 epochs.
2. Compare this augmented model to `full_model` on **the same fixed noisy test set**.
3. What happens to accuracy on the **clean** test set?

**Note:** higher robustness is not guaranteed; augmentation strength matters. Do not tune the noise amount on the test set in a real project.

In [ ]:
torch.manual_seed(SEED)
# BONUS TODO: train augmented_model with train_noise_std=0.25, then evaluate.
raise NotImplementedError("Train with noise augmentation and compare clean/noisy accuracy")

## Bonus 3 — Stop training at the right moment

The `fit` helper has an optional `patience` parameter. It monitors **validation loss**, stops after several non-improving epochs, and restores the best model weights.

1. Train a fresh model on `small_loader` for up to 80 epochs, using `patience=6`.
2. Plot validation loss, and compare the actual number of epochs with 80.
3. Explain why the test set must **not** be used for early stopping.

**Challenge within the bonus:** what changes if you monitor validation **accuracy** instead of validation loss?

In [ ]:
torch.manual_seed(SEED)
# BONUS TODO: fit an early-stopped small-data model (patience=6).
raise NotImplementedError("Experiment with early stopping")

## Wrap-up — discuss with your neighbor

1. What is the difference between a **training**, **validation**, and **test** set?
2. What happens when a model has more capacity than the available data can support?
3. Does dropout **always** help? Why should we compare actual measurements?
4. What is the difference between **testing on noisy inputs** and **training with noisy inputs**?

**Takeaway:** accuracy on clean training images is not the whole story. We care about **generalization** and **robustness**.